In [1]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df

,ID,propertyType,bedrooms,latitude,longitude,suburbName,distanceMetro(km),distanceAirport(km),distanceHospital(km),distanceRailway(km),area(square_meters),monthlyRent(us_dollar)
0,Train_0000,Apartment,3,28.638710,77.295822,Delhi East,0.312579,22.646032,11.726966,7.352495,83.61,307
1,Train_0001,Independent Floor,1,28.498940,77.207191,Delhi South,2.486167,13.500583,7.527761,15.877066,83.61,110
2,Train_0002,Independent Floor,3,28.714123,77.154404,Delhi North,1.528794,18.918243,17.135939,10.315737,78.97,369
3,Train_0003,Independent Floor,3,28.704330,77.149956,Other,0.967121,17.749252,16.251937,9.797817,162.58,676
4,Train_0004,Apartment,4,28.577915,77.049446,Dwarka,0.834506,4.288189,15.541840,18.179806,218.32,418
...,...,...,...,...,...,...,...,...,...,...,...,...
8687,Train_8687,Apartment,1,28.602234,77.026001,Dwarka,0.005681,7.776390,18.212199,19.535831,46.45,159
8688,Train_8688,Apartment,1,28.644989,77.169296,Delhi Central,0.007987,12.969368,9.442664,5.039023,81.29,172
8689,Train_8689,Independent Floor,3,28.547377,77.259155,Delhi South,0.203502,17.094466,5.468956,11.109941,148.64,738
8690,Train_8690,Independent Floor,1,28.630501,77.277382,Delhi East,0.248603,20.628700,9.801128,5.679541,41.90,184


In [2]:
drop_columns = ['propertyType', 'suburbName', 'ID']
label = 'monthlyRent(us_dollar)'

train_x = train_df.drop(drop_columns+[label], axis=1)
test_x_df = test_df.drop(drop_columns, axis=1)

train_y = train_df[label]


In [3]:
from sklearn.model_selection import train_test_split

train_x_df, val_x_df, train_y_df, val_y_df = \
    train_test_split(train_x, train_y, test_size=0.2, random_state=42)
    

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_x_scaled = scaler.fit_transform(train_x_df)
val_x_scaled = scaler.transform(val_x_df)
test_x_scaled = scaler.transform(test_x_df)

In [8]:
import torch

train_x_tensor = torch.tensor(train_x_scaled, dtype=torch.float)
val_x_tensor = torch.tensor(val_x_scaled, dtype=torch.float)
test_x_tensor = torch.tensor(test_x_scaled, dtype=torch.float)

train_y_tensor = torch.tensor(train_y_df.to_numpy(), dtype=torch.float)
val_y_tensor = torch.tensor(val_y_df.to_numpy(), dtype=torch.float)

print(train_x_tensor.shape, train_y_tensor.shape)
print(val_x_tensor.shape, val_y_tensor.shape)
print(test_x_tensor.shape)

torch.Size([6953, 8]) torch.Size([6953])
torch.Size([1739, 8]) torch.Size([1739])
torch.Size([8693, 8])


In [9]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(train_x_tensor, train_y_tensor)
val_dataset = TensorDataset(val_x_tensor, val_y_tensor)
test_dataset = TensorDataset(test_x_tensor)

type(train_dataset)

torch.utils.data.dataset.TensorDataset

In [10]:
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

type(train_loader)

torch.utils.data.dataloader.DataLoader

In [11]:
import torch
from torch.utils.data import Dataset

class CustomTensorDataset(torch.utils.data.Dataset):
    
    def __init__(self, data_tensors, target_tensors=None):
        self.data_tensors = data_tensors
        self.target_tensors = target_tensors

    def __len__(self):
        return self.data_tnsors.size(0)

    def __getitem__(self, index):
        data = self.data_tensors[index]
        if self.target_tensors is not None:
            target = self.target_tensors[index]
            return data, target
        return data

train_custom_dataset = CustomTensorDataset(train_x_tensor, train_y_tensor)
val_custom_dataset = CustomTensorDataset(val_x_tensor, val_y_tensor)
test_custom_dataset = CustomTensorDataset(test_x_tensor)

In [13]:
from torch.utils.data import DataLoader

train_loader2 = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_loader2 = DataLoader(val_dataset, batch_size=64, shuffle=False)

test_loader2 = DataLoader(test_dataset, batch_size=64, shuffle=False)

type(train_loader2)

torch.utils.data.dataloader.DataLoader